**Yandex maps**

In [23]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Настройки точки
network = 'Около'
address = 'Москва, Велозаводская ул., 9'
platform = 'Yandex'
url = 'https://yandex.by/maps/org/okolo/218784149504/reviews/?indoorLevel=1&ll=37.666540%2C55.712401&tab=reviews&z=17.06'

max_scroll = 20
out = 'all_reviews.xlsx'

options = webdriver.ChromeOptions()
options.add_argument('--window-size=1920,1080')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
reviews_list = []

try:
    driver.get(url)
    time.sleep(5)

    # Автоскролл страницы
    last_height = driver.execute_script('return document.body.scrollHeight')
    for _ in range(max_scroll):
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(4)
        new_height = driver.execute_script('return document.body.scrollHeight')
        if new_height == last_height:
            break
        last_height = new_height

    # Сбор данных
    reviews = driver.find_elements(By.CLASS_NAME, 'business-reviews-card-view__review')
    for review in reviews:
        date_el = review.find_elements(By.CLASS_NAME, 'business-review-view__date')
        date = date_el[0].text if date_el else 'Нет даты'

        stars_blocks = review.find_elements(By.CLASS_NAME, 'business-rating-badge-view__stars')
        rating = 'Не удалось определить'
        if stars_blocks:
            full_stars = stars_blocks[0].find_elements(By.CLASS_NAME, '_full')
            if full_stars:
                rating = len(full_stars)

        text_el = review.find_elements(By.CLASS_NAME, 'spoiler-view__text-container')
        text = text_el[0].text if text_el else 'Пустой отзыв'

        reviews_list.append({
            'Сеть / формат': network,
            'Адрес точки': address,
            'Источник': platform,
            'Текст отзыва': text,
            'Оценка': rating,
            'Дата': date})

    # Сохранение и объединение данных
    if reviews_list:
        new_df = pd.DataFrame(reviews_list)
        print(f'Скачано - {len(new_df)}')
        if os.path.exists(out):
            old_df = pd.read_excel(out)
            if '№' in old_df.columns:
                old_df = old_df.drop(columns=['№'])
            final_df = pd.concat([old_df, new_df], ignore_index=True)
        else:
            final_df = new_df

        # Чистка дубликатов и создание сквозного индекса
        final_df = final_df.drop_duplicates(subset=['Адрес точки', 'Текст отзыва', 'Дата'], keep='first')
        final_df.insert(0, '№', range(1, len(final_df) + 1))
        
        final_df.to_excel(out, index=False)

except Exception as e:
    print(f'Ошибка: {e}')

finally:
    driver.quit()

Скачано - 50


**VC.RU**

In [ ]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

network = 'Дикси Go'
address = 'Не указан'
platform = 'vc.ru'
url = 'https://vc.ru/retail/1726205-dixygo'

max_scroll = 5
out = 'all_reviews.xlsx'

options = webdriver.ChromeOptions()
options.add_argument('--window-size=1920,1080')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
reviews_list = []

try:
    driver.get(url)
    time.sleep(5)

    time.sleep(3)
    expand_buttons = driver.find_elements(By.CLASS_NAME, 'comments-limit__expand')
    
    if expand_buttons:
        try:
            btn = expand_buttons[0]
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(2)
            driver.execute_script('arguments[0].click();', btn)
            print('Все комментарии развернуты')
            time.sleep(5)
        except Exception as click_err:
            print(f'Комментарии не раскрылись {click_err}')

    last_height = driver.execute_script('return document.body.scrollHeight')
    for _ in range(max_scroll):
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(4)
        new_height = driver.execute_script('return document.body.scrollHeight')
        if new_height == last_height:
            break
        last_height = new_height

    # ограничение только блоком текущей статьи
    core_containers = driver.find_elements(By.CLASS_NAME, 'comments-core')
    
    if core_containers:
        comments = core_containers[0].find_elements(By.CLASS_NAME, 'comment--root')
    else:
        comments = driver.find_elements(By.CLASS_NAME, 'comment--root')
    
    for comment in comments:
        # Текст комментария
        text_el = comment.find_elements(By.CLASS_NAME, 'comment-text')
        text = text_el[0].text if text_el else 'Пустой комментарий'
        
        if text == 'Пустой комментарий' or not text.strip():
            continue

        # Дата комментария
        time_el = comment.find_elements(By.TAG_NAME, 'time')
        if time_el:
            date = time_el[0].get_attribute('title')
            if not date:
                date = time_el[0].text
        else:
            date = 'Нет даты'

        rating = 'Форум (без оценки)'

        reviews_list.append({
            'Сеть / формат': network,
            'Адрес точки': address,
            'Источник': platform,
            'Текст отзыва': text,
            'Оценка': rating,
            'Дата': date
        })

    # Сохранение и объединение данных
    if reviews_list:
        new_df = pd.DataFrame(reviews_list)
        print(f'Скачано - {len(new_df)}')
        
        if os.path.exists(out):
            old_df = pd.read_excel(out)
            if '№' in old_df.columns:
                old_df = old_df.drop(columns=['№'])
            final_df = pd.concat([old_df, new_df], ignore_index=True)
        else:
            final_df = new_df

        final_df = final_df.drop_duplicates(subset=['Источник', 'Адрес точки', 'Текст отзыва', 'Дата'], keep='first')
        final_df.insert(0, '№', range(1, len(final_df) + 1))
        
        final_df.to_excel(out, index=False)

except Exception as e:
    print(f'Ошибка: {e}')

finally:
    driver.quit()

**ХАБР**

In [6]:
import os
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

network = 'Налету'
address = 'Не указан'
platform = 'Habr'
url = 'https://habr.com/ru/companies/X5Tech/articles/542682/comments/'  

max_scroll = 5
out = 'all_reviews.xlsx'

options = webdriver.ChromeOptions()
options.add_argument('--window-size=1920,1080')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
reviews_list = []


def clean_date(raw_date):
    '''Превращает '16 фев 2021 в 14:51' в '16 фев 2021''''
    if not raw_date:
        return 'Нет даты'
    cleaned = re.sub(r'\s+в\s+\d{2}:\d{2}.*$', '', raw_date)
    return cleaned.replace('\xa0', ' ').strip()


try:
    driver.get(url)
    time.sleep(5)

    last_height = driver.execute_script('return document.body.scrollHeight')
    for _ in range(max_scroll):
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(3)
        new_height = driver.execute_script('return document.body.scrollHeight')
        if new_height == last_height:
            break
        last_height = new_height

    root_threads = driver.find_elements(
        By.CLASS_NAME, 'tm-comment-thread__indent_l-0'
    )
    print(f'Найдено корневых комментариев: {len(root_threads)}')

    for thread in root_threads:
        try:
            # Ищем тело комментария внутри корневого треда
            body_el = thread.find_elements(
                By.CLASS_NAME, 'tm-comment__body-content'
            )
            if not body_el:
                continue

            comment_body = body_el[0]

            # удалить цитату
            text = driver.execute_script(
                '''
                var clone = arguments[0].cloneNode(true);
                var quotes = clone.getElementsByTagName('blockquote');
                while(quotes.length > 0) {
                    quotes[0].parentNode.removeChild(quotes[0]);
                }
                return clone.innerText;
            ''', comment_body,)

            if not text or not text.strip():
                continue

            # Сбор даты
            time_el = thread.find_elements(By.TAG_NAME, 'time')
            if time_el:
                raw_date = time_el[0].text  
                date = clean_date(raw_date)
            else:
                date = 'Нет даты'

            rating = 'Форум (без оценки)'

            reviews_list.append(
                {
                    'Сеть / формат': network,
                    'Адрес точки': address,
                    'Источник': platform,
                    'Текст отзыва': text.strip(),
                    'Оценка': rating,
                    'Дата': date,
                }
            )
        except Exception as item_err:
            print(f'Ошибка при обработке комментария: {item_err}')
            continue

    # сохранение и обьединение данных
    if reviews_list:
        new_df = pd.DataFrame(reviews_list)
        print(f'Успешно извлечено корневых отзывов: {len(new_df)}')

        if os.path.exists(out):
            old_df = pd.read_excel(out)
            if '№' in old_df.columns:
                old_df = old_df.drop(columns=['№'])
            final_df = pd.concat([old_df, new_df], ignore_index=True)
        else:
            final_df = new_df

        # Удаляем дубликаты
        final_df = final_df.drop_duplicates(
            subset=['Источник', 'Адрес точки', 'Текст отзыва', 'Дата'],
            keep='first',
        )
        final_df.insert(0, '№', range(1, len(final_df) + 1))

        final_df.to_excel(out, index=False)
        print('Данные успешно сохранены в файл.')
    else:
        print('Корневые комментарии не найдены или они пустые.')

except Exception as e:
    print(f'Глобальная ошибка: {e}')

finally:
    driver.quit()

Найдено корневых комментариев: 51
Успешно извлечено корневых отзывов: 47
Данные успешно сохранены в файл.
